In [6]:
import os
from IPython.display import display, Markdown  
import dspy
from dotenv import load_dotenv

# override=True: the kernel keeps the first-loaded values in os.environ, so a plain
# load_dotenv() silently ignores an edited .env until the kernel is restarted.
load_dotenv(override=True)

# Local Ollama — no API key, no free-tier quota. Use `ollama_chat/` (not `ollama/`):
# it routes through /api/chat so the model's own chat template is applied.
dspy.configure(
    lm=dspy.LM(
        "ollama_chat/" + os.environ.get("OLLAMA_MODEL", "llama3:8b"),
        api_base=os.environ.get("OLLAMA_API_BASE", "http://localhost:11434"),
        api_key="",
    )
)

haiku_signature = "subject -> haiku"
haiku_generator = dspy.Predict(haiku_signature)
result = haiku_generator(subject="computer science")
display(Markdown(f"**Haiku:** {result.haiku}"))


**Haiku:** Codes dance in silence
Problem-solving whispers
Logic's gentle art

In [30]:
poem_signature = "topic -> poem"
poem_generator = dspy.Predict(poem_signature)
result = poem_generator(topic="python vs rust")
display(Markdown(f"**Poem:** {result.poem}"))

**Poem:** One slithers smooth in readable lines,
Where whitespace shapes the clear designs.
A friendly snake for swift creation,
With dynamic grace in automation.
Just "import" and the scripts take flight,
Though runtime pauses in the night.

The other forged in iron and flame,
Where Ferris plays a stricter game.
The borrow checker guards the gate,
No null, no leaks, no race-prone fate.
A steeper climb, a battle won,
To see bare-metal speed outrun.

One trades the clock for ease and grace,
One claims the crown in memory’s race.
The serpent dreams in simple flow,
While gears of Rust blaze fast below.

In [9]:
comparison_signature = "language_a, language_b -> comparison"

class DspyObj(dspy.Signature):
    comparison: str = dspy.OutputField()
    language_a: str = dspy.InputField()
    language_b: str = dspy.InputField()
compare = dspy.Predict(DspyObj)

result = compare(
    language_a="Python",
    language_b="Rust",
)

print(result.comparison)

Python vs Rust: A Comparison


In [3]:
from typing  import Protocol


class Writable(Protocol):
    def  write(self, data: str) -> None:
        ...

class Readable(Protocol):
    def read(self) -> str:
        ...

def do_write(write: Writable, data: str) -> None:
    write.write(data)

def do_read(read: Readable) -> str:
    return read.read()

class Author(Writable, Readable):
    def __init__(self, name: str):
        self.name = name

    def write(self, data: str) -> None:
        print(f"{self.name} is writing: {data}")

    def read(self) -> str:
        return f"{self.name} is reading."

def main():
    data: dict = {"name": "Alice", "data": "Hello, World!"}
    author = Author("Alice")
    do_write(author, data["data"])
    print(do_read(author))
if __name__ == "__main__":
    main()

Alice is writing: Hello, World!
Alice is reading.


In [ ]:
from dspy import MultiChainComparison

In [ ]:
# Swap models here: any tag from `ollama list` works, e.g. "qwen3:8b", "mistral:7b".
dspy.configure(
    lm=dspy.LM(
        "ollama_chat/" + os.environ.get("OLLAMA_MODEL", "llama3:8b"),
        api_base=os.environ.get("OLLAMA_API_BASE", "http://localhost:11434"),
        api_key="",
    )
)

In [ ]:
import os

from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv(override=True)

llm = ChatOllama(
    model=os.environ.get("OLLAMA_MODEL", "llama3:8b"),
    base_url=os.environ.get("OLLAMA_API_BASE", "http://localhost:11434"),
    temperature=0,
)

# No retry wrapper needed here: a local server has no 503 "high demand" or 429 quota.
# The one failure mode is a cold start — the first call blocks while the weights load.
response = llm.invoke("Say hello in one sentence.")

print(response.text)  # .content is a list of blocks; .text is the plain string


In [8]:
history = dspy.inspect_history()
print(f"History has {history} entries.")





[2026-08-16T22:50:25.261781]

System message:

Your input fields are:
1. `language_a` (str): 
2. `language_b` (str):
Your output fields are:
1. `comparison` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## language_a ## ]]
{language_a}

[[ ## language_b ## ]]
{language_b}

[[ ## comparison ## ]]
{comparison}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `language_a`, `language_b`, produce the fields `comparison`.


User message:

[[ ## language_a ## ]]
Python

[[ ## language_b ## ]]
Rust

Respond with the corresponding output fields, starting with the field `[[ ## comparison ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## comparison ## ]]
Python vs Rust: A Comparison

[[ ## completed ## ]]





History has None entries.


In [12]:
from __future__ import annotations
import typing

class Node:
    data: int
    next: typing.Optional[Node]

print(typing.get_type_hints(Node))

{'data': <class 'int'>, 'next': __main__.Node | None}
